In [1]:
import sys
!{sys.executable} -m pip install yfinance

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import sys
sys.path.insert(0, "..")

from src.data.price_loader import download_prices, get_log_returns

prices = download_prices("AAPL", "2023-01-01", "2023-03-31")
print(prices.columns.tolist())
print(prices.shape)

returns = get_log_returns(prices)
print(returns.head())

['Close', 'High', 'Low', 'Open', 'Volume']
(61, 5)
Date
2023-01-04    0.010261
2023-01-05   -0.010661
2023-01-06    0.036133
2023-01-09    0.004080
2023-01-10    0.004447
Name: Close, dtype: float64


In [3]:
import pandas as pd
from src.data.event_builder import (
    UNIVERSE, ANALYSIS_START, ANALYSIS_END,
    get_pre_window, get_post_window, compute_abnormal_return
)
from src.data.price_loader import download_prices, get_log_returns

# Quick sanity check on windowing
prices = download_prices("AAPL", "2022-01-01", "2023-06-30")
returns = get_log_returns(prices)

# Simulate an event on 2023-02-02 (AAPL Q4 2022 earnings)
event_date = pd.Timestamp("2023-02-02")
pre  = get_pre_window(returns, event_date, window=30)
post = get_post_window(returns, event_date, window=5)

print("Pre window:", pre.index[0].date(), "→", pre.index[-1].date(), f"({len(pre)} days)")
print("Post window:", post.index[0].date(), "→", post.index[-1].date(), f"({len(post)} days)")
print("Overlap check (must be 0):", len(set(pre.index) & set(post.index)))

Pre window: 2022-12-19 → 2023-02-01 (30 days)
Post window: 2023-02-02 → 2023-02-08 (5 days)
Overlap check (must be 0): 0


In [4]:
import sys
import numpy as np
sys.path.insert(0, "..")

from src.fourier.spectral import compute_spectral_features

# Use the pre-window we already computed
features = compute_spectral_features(pre.values)

print(f"Spectral entropy:   {features['spectral_entropy']:.4f}")
print(f"Dominant frequency: {features['dominant_frequency']:.4f}")
print(f"Dominant power:     {features['dominant_power']:.4f}")
print(f"N observations:     {features['n_obs']}")

Spectral entropy:   2.4819
Dominant frequency: 0.3667
Dominant power:     0.2130
N observations:     30


In [5]:
from src.fourier.features import extract_features_for_event
from src.data.price_loader import download_prices, get_log_returns

spy_prices  = download_prices("SPY", "2022-01-01", "2023-06-30")
spy_returns = get_log_returns(spy_prices)

result = extract_features_for_event(
    ticker="AAPL",
    event_date=pd.Timestamp("2023-02-02"),
    stock_returns=returns,
    benchmark_returns=spy_returns,
)

print(result)

{'ticker': 'AAPL', 'event_date': Timestamp('2023-02-02 00:00:00'), 'spectral_entropy': np.float64(2.481869841110682), 'dominant_frequency': np.float64(0.36666666666666664), 'dominant_power': np.float64(0.2130066982729888), 'cumulative_abnormal_return': np.float64(0.04402431056313236), 'day1_abnormal_return': np.float64(0.021940221853018388), 'pre_window_size': 30, 'post_window_size': 5}


In [6]:
import sys
!{sys.executable} -m pip install lxml

import yfinance as yf
ticker = yf.Ticker("AAPL")
dates = ticker.earnings_dates
print(dates)
print(dates.shape)

Defaulting to user installation because normal site-packages is not writeable
                           EPS Estimate  Reported EPS  Surprise(%)
Earnings Date                                                     
2026-04-30 16:00:00-04:00          1.94           NaN          NaN
2026-01-29 16:00:00-05:00          2.67          2.84         6.25
2025-10-30 16:00:00-04:00          1.77          1.85         4.52
2025-07-31 16:00:00-04:00          1.43          1.57         9.48
2025-05-01 16:00:00-04:00          1.63          1.65         1.50
2025-01-30 16:00:00-05:00          2.35          2.40         2.26
2024-10-31 16:00:00-04:00          1.60          1.64         2.28
2024-08-01 16:00:00-04:00          1.34          1.40         4.30
2024-05-02 16:00:00-04:00          1.51          1.53         1.46
2024-02-01 16:00:00-05:00          2.10          2.18         3.66
2023-11-02 16:00:00-04:00          1.39          1.46         4.82
2023-08-03 16:00:00-04:00          1.19          1.

In [7]:
import sys
sys.path.insert(0, "..")

from src.data.price_loader import fetch_all_earnings_dates
from src.data.event_builder import UNIVERSE

events = fetch_all_earnings_dates(UNIVERSE)
print(events.shape)
print(events.head(10))
print("\nEvents per ticker:")
print(events.groupby("ticker").size().sort_values())

Fetching AAPL...
Fetching MSFT...
Fetching GOOGL...
Fetching AMZN...
Fetching META...
Fetching NVDA...
Fetching AMD...
Fetching INTC...
Fetching TSLA...
Fetching NFLX...
Fetching ORCL...
Fetching CRM...
Fetching ADBE...
Fetching QCOM...
Fetching TXN...

Saved 375 earnings events to D:\MathForDevs\Data Science\earnings-signals\data\processed\earnings_dates.csv
(375, 5)
  ticker earnings_date  eps_estimate  reported_eps  surprise_pct
0   AAPL    2026-04-30          1.94           NaN           NaN
1   AAPL    2026-01-29          2.67          2.84          6.25
2   AAPL    2025-10-30          1.77          1.85          4.52
3   AAPL    2025-07-31          1.43          1.57          9.48
4   AAPL    2025-05-01          1.63          1.65          1.50
5   AAPL    2025-01-30          2.35          2.40          2.26
6   AAPL    2024-10-31          1.60          1.64          2.28
7   AAPL    2024-08-01          1.34          1.40          4.30
8   AAPL    2024-05-02          1.51        